# Stable Diffusion 1.5 Ultimate Workstation
This notebook has been upgraded to a full-featured AI Art Station:
- **Modes**: Text-to-Image & Image-to-Image.
- **Universal Loader**: CivitAI & Hugging Face support.
- **Multi-LoRA**: Load up to 3 LoRAs simultaneously.
- **Wildcards**: Dynamic prompts using `__wildcard__` syntax.
- **AI Upscaling**: Fast (FSRCNN) & Quality (EDSR) options.
- **Performance**: Optimized for low RAM/VRAM usage.
- **Persistence**: Prompts saved in browser cache automatically.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors gradio omegaconf invisible-watermark opencv-contrib-python-headless huggingface_hub

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler
import os
import requests
from datetime import datetime
from google.colab import drive
from PIL import Image
import json
import random
import shutil
import cv2
import numpy as np
import re
from huggingface_hub import login

# --- Global Configuration ---
HISTORY_DIR = "/content/sd_history"
DRIVE_ROOT = "/content/drive/MyDrive"
WILDCARDS_DIR = "/content/wildcards"
MODELS_CACHE = "/content/models_cache"

# Ensure directories
for d in [HISTORY_DIR, WILDCARDS_DIR, MODELS_CACHE]:
    os.makedirs(d, exist_ok=True)

# Create sample wildcard
if not os.path.exists(os.path.join(WILDCARDS_DIR, "colors.txt")):
    with open(os.path.join(WILDCARDS_DIR, "colors.txt"), "w") as f:
        f.write("red\nblue\ngreen\nyellow\npurple\ncyan\nmagenta\ngold\nsilver\nneon pink")

# --- Global State ---
pipe_t2i = None
pipe_i2i = None
current_model_path = ""
drive_mounted = False
loaded_loras = {} # {path: adapter_name}
upscaler_edsr = None
upscaler_fsrcnn = None

# --- Javascript for Persistence ---
js_persistence = """
function() {
    // Restore values
    const fields = ['txt_prompt', 'txt_neg_prompt'];
    fields.forEach(id => {
        const val = localStorage.getItem('sd_' + id);
        if (val) {
            const el = document.querySelector('#' + id + ' textarea');
            if (el) { 
                el.value = val; 
                el.dispatchEvent(new Event('input', { bubbles: true }));
            }
        }
    });

    // Attach listeners to save values
    fields.forEach(id => {
        const el = document.querySelector('#' + id + ' textarea');
        if (el) {
            el.addEventListener('input', (e) => {
                localStorage.setItem('sd_' + id, e.target.value);
            });
        }
    });
}
"""

# --- Helper Classes ---
class WildcardManager:
    def process(self, prompt):
        if not prompt: return prompt
        
        def replace_match(match):
            wname = match.group(1)
            wpath = os.path.join(WILDCARDS_DIR, f"{wname}.txt")
            if os.path.exists(wpath):
                with open(wpath, 'r') as f:
                    lines = [line.strip() for line in f if line.strip()]
                if lines:
                    return random.choice(lines)
            return match.group(0) # Keep if not found

        # Iterate until no more replacements (allows nested wildcards)
        for _ in range(3):
            new_prompt = re.sub(r"__([a-zA-Z0-9_\-\s]+)__", replace_match, prompt)
            if new_prompt == prompt:
                break
            prompt = new_prompt
        return prompt

wildcard_manager = WildcardManager()

class PromptManager:
    def __init__(self):
        self.local_path = "/content/saved_prompts.json"
        self.drive_path = os.path.join(DRIVE_ROOT, "SD_Outputs", "saved_prompts.json")

    def get_filepath(self):
        if os.path.exists(DRIVE_ROOT):
            drive_dir = os.path.dirname(self.drive_path)
            if not os.path.exists(drive_dir):
                try: os.makedirs(drive_dir)
                except: pass
            if os.path.exists(self.local_path) and not os.path.exists(self.drive_path):
                try: shutil.copy(self.local_path, self.drive_path)
                except: pass
            return self.drive_path
        return self.local_path

    def load(self):
        filepath = self.get_filepath()
        if not os.path.exists(filepath): return {}
        try:
            with open(filepath, 'r') as f: return json.load(f)
        except: return {}

    def save(self, name, prompt, neg_prompt):
        data = self.load()
        data[name] = {"prompt": prompt, "neg_prompt": neg_prompt}
        with open(self.get_filepath(), 'w') as f: json.dump(data, f)
        return gr.update(choices=list(data.keys()))

    def get_prompts_list(self): return list(self.load().keys())
    def get_prompt(self, name):
        data = self.load()
        if name in data: return data[name]["prompt"], data[name]["neg_prompt"]
        return "", ""

prompt_manager = PromptManager()

# --- Core Functions ---
def mount_google_drive():
    global drive_mounted
    if not drive_mounted:
        try:
            drive.mount('/content/drive')
            drive_mounted = True
            return True
        except Exception as e:
            print(f"Error mounting drive: {e}")
            return False
    return True

def download_file(url, filename, subdir=None):
    target_dir = MODELS_CACHE
    if subdir:
        target_dir = os.path.join(MODELS_CACHE, subdir)
        os.makedirs(target_dir, exist_ok=True)
        
    print(f"Downloading {filename} from {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    final_name = filename
    if "content-disposition" in response.headers and filename.startswith("custom"):
        import re
        fname = re.findall("filename=(.+)", response.headers["content-disposition"])
        if fname:
            extracted_name = fname[0].strip('"')
            final_name = os.path.basename(extracted_name)

    final_path = os.path.join(target_dir, final_name)
    if os.path.exists(final_path):
        print(f"File already exists: {final_path}")
        return final_path

    with open(final_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded to {final_path}")
    return final_path

def setup_upscaler(method="fsrcnn"):
    global upscaler_edsr, upscaler_fsrcnn
    from cv2 import dnn_superres
    
    if method == "edsr":
        if upscaler_edsr: return upscaler_edsr
        path = os.path.join(MODELS_CACHE, "EDSR_x2.pb")
        if not os.path.exists(path):
            download_file("https://github.com/Saafke/EDSR_Tensorflow/raw/master/models/EDSR_x2.pb", "EDSR_x2.pb")
        upscaler_edsr = dnn_superres.DnnSuperResImpl_create()
        upscaler_edsr.readModel(path)
        upscaler_edsr.setModel("edsr", 2)
        return upscaler_edsr
    else: # fsrcnn (fast)
        if upscaler_fsrcnn: return upscaler_fsrcnn
        path = os.path.join(MODELS_CACHE, "FSRCNN_x2.pb")
        if not os.path.exists(path):
            download_file("https://github.com/Saafke/FSRCNN_Tensorflow/raw/master/models/FSRCNN_x2.pb", "FSRCNN_x2.pb")
        upscaler_fsrcnn = dnn_superres.DnnSuperResImpl_create()
        upscaler_fsrcnn.readModel(path)
        upscaler_fsrcnn.setModel("fsrcnn", 2)
        return upscaler_fsrcnn

def upscale_image_cv2(image_pil, method="fsrcnn"):
    try:
        model = setup_upscaler(method)
        img = np.array(image_pil)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        result = model.upsample(img)
        result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        return Image.fromarray(result)
    except Exception as e:
        print(f"Upscale error: {e}")
        return image_pil

def load_model(model_path_or_url, hf_token):
    global pipe_t2i, pipe_i2i, current_model_path, loaded_loras
    
    if hf_token.strip():
        try: login(token=hf_token)
        except: pass

    if pipe_t2i is not None and model_path_or_url == current_model_path:
        return "Model already loaded."

    print(f"Loading model: {model_path_or_url}...")
    
    try:
        final_path = model_path_or_url
        is_repo = not model_path_or_url.startswith("http") and "/" in model_path_or_url
        
        if not is_repo:
            if model_path_or_url.startswith("http"):
                final_path = download_file(model_path_or_url, "custom_model.safetensors")

        if is_repo:
            pipe_t2i = StableDiffusionPipeline.from_pretrained(
                final_path, torch_dtype=torch.float16, use_safetensors=True,
                use_auth_token=hf_token if hf_token else True
            )
        else:
            pipe_t2i = StableDiffusionPipeline.from_single_file(
                final_path, torch_dtype=torch.float16
            )

        pipe_t2i.scheduler = DPMSolverMultistepScheduler.from_config(
            pipe_t2i.scheduler.config, use_karras_sigmas=True, algorithm_type="dpmsolver++"
        )
        
        # Optimizations for RAM/VRAM
        pipe_t2i.enable_model_cpu_offload()
        pipe_t2i.vae.enable_tiling() # Reduce VRAM peak
        pipe_t2i.enable_attention_slicing() # Reduce RAM peak
        pipe_t2i.safety_checker = None
        
        # Shared components
        pipe_i2i = StableDiffusionImg2ImgPipeline(vae=pipe_t2i.vae, text_encoder=pipe_t2i.text_encoder, tokenizer=pipe_t2i.tokenizer, unet=pipe_t2i.unet, scheduler=pipe_t2i.scheduler, safety_checker=None, feature_extractor=None)
        
        current_model_path = model_path_or_url
        loaded_loras = {} # Reset LoRAs on model change
        return "Model loaded successfully!"
    except Exception as e:
        return f"Error loading model: {str(e)}"

def apply_loras(lora1, scale1, lora2, scale2, lora3, scale3):
    global pipe_t2i, loaded_loras
    if pipe_t2i is None: return "Load model first."
    
    active_adapters = []
    active_scales = []
    status = []

    inputs = [(lora1, scale1), (lora2, scale2), (lora3, scale3)]
    
    for i, (path, scale) in enumerate(inputs):
        if not path: continue
        
        adapter_name = f"lora_{i}"
        final_path = path
        
        # Download if needed
        if path.startswith("http"):
            final_path = download_file(path, f"custom_lora_{i}.safetensors", subdir="loras")
        
        # Load if not already loaded
        if loaded_loras.get(final_path) != adapter_name:
            try:
                print(f"Loading LoRA {i+1}...")
                pipe_t2i.load_lora_weights(final_path, adapter_name=adapter_name)
                loaded_loras[final_path] = adapter_name
            except Exception as e:
                status.append(f"Error LoRA {i+1}: {e}")
                continue
        
        active_adapters.append(adapter_name)
        active_scales.append(float(scale))
        status.append(f"LoRA {i+1} Active")

    if active_adapters:
        pipe_t2i.set_adapters(active_adapters, adapter_weights=active_scales)
    else:
        pipe_t2i.unload_lora_weights()
        loaded_loras = {}
        status.append("LoRAs cleared")
        
    return ", ".join(status)

def generate(mode, prompt, neg_prompt, img_input, steps, cfg, width, height, seed, randomize, num_images, save_drive, drive_folder, denoise, do_upscale, upscale_method):
    global pipe_t2i, pipe_i2i
    if pipe_t2i is None:
        return [None] * int(num_images), seed

    if randomize:
        seed = random.randint(0, 2147483647)
    
    # Process Wildcards
    prompt_processed = wildcard_manager.process(prompt)
    neg_prompt_processed = wildcard_manager.process(neg_prompt)
    
    print(f"Generating {num_images} images | Prompt: {prompt_processed[:50]}...")
    
    generated_images = []
    
    for i in range(int(num_images)):
        current_seed = seed + i if not randomize else random.randint(0, 2147483647)
        if i == 0 and not randomize: current_seed = seed
        
        generator = torch.Generator(device="cpu").manual_seed(int(current_seed))

        if mode == "txt2img":
            out = pipe_t2i(
                prompt_processed, negative_prompt=neg_prompt_processed, 
                num_inference_steps=steps, guidance_scale=cfg, 
                width=width, height=height, num_images_per_prompt=1, 
                generator=generator
            ).images[0]
        else: # img2img
            if img_input is None:
                continue
            init_image = img_input.resize((width, height))
            out = pipe_i2i(
                prompt_processed, image=init_image, negative_prompt=neg_prompt_processed, 
                num_inference_steps=steps, guidance_scale=cfg, strength=denoise,
                num_images_per_prompt=1, generator=generator
            ).images[0]
        
        if do_upscale:
            out = upscale_image_cv2(out, upscale_method)
            
        generated_images.append(out)

    # Save Logic
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    for i, img in enumerate(generated_images):
        fname = f"sd_{timestamp}_{i}.png"
        img.save(os.path.join(HISTORY_DIR, fname))
    
    if save_drive and mount_google_drive():
        drive_path = f"/content/drive/MyDrive/{drive_folder}"
        os.makedirs(drive_path, exist_ok=True)
        for i, img in enumerate(generated_images):
            img.save(f"{drive_path}/sd_{timestamp}_{i}.png")

    return generated_images, seed

def get_history_images():
    files = sorted(os.listdir(HISTORY_DIR), reverse=True)
    paths = [os.path.join(HISTORY_DIR, f) for f in files if f.endswith('.png')]
    return paths

def select_image(evt: gr.SelectData):
    return evt.value['image']['path']

def delete_image(path):
    if not path: return get_history_images(), "No image selected."
    try:   
        filename = os.path.basename(path)
        real_path = os.path.join(HISTORY_DIR, filename)
        if os.path.exists(real_path):
            os.remove(real_path)
            return get_history_images(), "Image deleted."
        return get_history_images(), "Image not found."
    except Exception as e:
        return get_history_images(), f"Error: {str(e)}"

# --- UI Layout ---
with gr.Blocks(theme=gr.themes.Soft(), title="SD Ultimate Workstation") as demo:
    gr.Markdown("# Stable Diffusion 1.5 Ultimate Workstation")
    
    with gr.Row():
        # --- Left: Inputs ---
        with gr.Column(scale=1):
            # 1. Model & LoRA
            with gr.Accordion("1. Model & LoRA", open=True):
                with gr.Row():
                    model_input = gr.Textbox(label="Base Model", value="runwayml/stable-diffusion-v1-5", scale=2)
                    hf_token = gr.Textbox(label="HF Token", type="password", scale=1)
                load_model_btn = gr.Button("Load Model", variant="secondary", size="sm")
                
                gr.Markdown("**LoRA Slots**")
                with gr.Row():
                    l1 = gr.Textbox(label="LoRA 1 URL", scale=3)
                    s1 = gr.Slider(label="Scale", min=0, max=1, value=0.7, scale=1)
                with gr.Row():
                    l2 = gr.Textbox(label="LoRA 2 URL", scale=3)
                    s2 = gr.Slider(label="Scale", min=0, max=1, value=0.7, scale=1)
                with gr.Row():
                    l3 = gr.Textbox(label="LoRA 3 URL", scale=3)
                    s3 = gr.Slider(label="Scale", min=0, max=1, value=0.7, scale=1)
                apply_lora_btn = gr.Button("Apply LoRAs", size="sm")
                status_text = gr.Textbox(label="Status", interactive=False, max_lines=1)

            # 2. Generation
            with gr.Group():
                gr.Markdown("### 2. Prompt & Generate")
                # Mode Tabs
                mode_state = gr.State(value="txt2img")
                with gr.Tabs() as mode_tabs:
                    with gr.Tab("Text-to-Image", id="t2i_tab"): pass
                    with gr.Tab("Image-to-Image", id="i2i_tab"):
                        # Updated to use sources parameter for better compatibility
                        img_input = gr.Image(label="Input Image", type="pil", height=250, sources=["upload", "clipboard", "webcam"])
                        denoise = gr.Slider(label="Denoise Strength", minimum=0.0, maximum=1.0, value=0.75)

                prompt = gr.Textbox(label="Prompt", lines=3, elem_id="txt_prompt")
                neg_prompt = gr.Textbox(label="Negative Prompt", lines=2, value="low quality, bad anatomy, worst quality", elem_id="txt_neg_prompt")
                
                with gr.Row():
                     gen_btn = gr.Button("GENERATE", variant="primary", size="lg", scale=2)
                     num_images = gr.Slider(label="Count", minimum=1, maximum=10, step=1, value=1, scale=1)

            # 3. Settings
            with gr.Accordion("Advanced Settings", open=True):
                with gr.Row():
                    width = gr.Slider(label="Width", minimum=256, maximum=1024, step=64, value=512)
                    height = gr.Slider(label="Height", minimum=256, maximum=1024, step=64, value=512)
                with gr.Row():
                    steps = gr.Slider(label="Steps", minimum=10, maximum=100, step=1, value=25)
                    cfg = gr.Slider(label="CFG", minimum=1, maximum=20, step=0.5, value=7.5)
                with gr.Row():
                    random_seed = gr.Checkbox(label="Random Seed", value=True)
                    seed = gr.Number(label="Seed", value=42, precision=0)
                
                with gr.Row():
                    do_upscale = gr.Checkbox(label="AI Upscale (2x)", value=False)
                    upscale_method = gr.Radio(label="Method", choices=["fsrcnn", "edsr"], value="fsrcnn", info="FSRCNN=Fast, EDSR=Quality")
                
                with gr.Row():
                    save_drive = gr.Checkbox(label="Save to Drive", value=False)
                    drive_folder = gr.Textbox(label="Folder", value="SD_Outputs", show_label=False)

            with gr.Accordion("Prompt Manager", open=False):
                style_name = gr.Textbox(label="Name")
                save_style_btn = gr.Button("Save Current")
                saved_styles_drop = gr.Dropdown(label="Load Style", choices=prompt_manager.get_prompts_list())
                load_style_btn = gr.Button("Load Selected")

        # --- Right: Results ---
        with gr.Column(scale=1):
            gallery = gr.Gallery(label="Output", columns=2, height=600)
            with gr.Row():
                refresh_hist_btn = gr.Button("Refresh History")
                del_img_btn = gr.Button("Delete Selected", variant="stop")
            hist_gallery = gr.Gallery(label="History", columns=4, allow_preview=True, height=300)
            hist_msg = gr.Textbox(label="System Info", show_label=False)
            selected_img_path = gr.State(value="")

    # --- Wiring ---
    load_model_btn.click(load_model, [model_input, hf_token], [status_text])
    apply_lora_btn.click(apply_loras, [l1, s1, l2, s2, l3, s3], [status_text])

    t2i_tab = mode_tabs.children[0]
    i2i_tab = mode_tabs.children[1]
    t2i_tab.select(fn=lambda: "txt2img", outputs=mode_state)
    i2i_tab.select(fn=lambda: "img2img", outputs=mode_state)

    gen_btn.click(
        fn=generate,
        inputs=[mode_state, prompt, neg_prompt, img_input, steps, cfg, width, height, seed, random_seed, num_images, save_drive, drive_folder, denoise, do_upscale, upscale_method],
        outputs=[gallery, seed]
    )

    refresh_hist_btn.click(get_history_images, outputs=[hist_gallery])
    hist_gallery.select(select_image, None, selected_img_path)
    del_img_btn.click(delete_image, inputs=[selected_img_path], outputs=[hist_gallery, hist_msg])

    save_style_btn.click(lambda n,p,np: prompt_manager.save(n,p,np), [style_name, prompt, neg_prompt], [saved_styles_drop])
    load_style_btn.click(lambda n: prompt_manager.get_prompt(n), [saved_styles_drop], [prompt, neg_prompt])
    
    # Inject JS for persistence
    demo.load(None, None, None, _js=js_persistence)

demo.launch(share=True, debug=True)